# Run T03

Dynamic scenario, ramps (ID 5127). Window 2026-07-30 13:15:00 to 2026-07-30 14:46:00, output interval 0.01 s.

Model: VFCS `5fd49f1` with `subSystemAir = SubSystemAirCMP` and `subSystemCooling = SubSystemCoolingTST` in `FuelCellSystem`. Solver DASSL, tolerance 1e-4.

Workflow: measured data, simulation input, stored simulation result, comparison on the measured time base, evaluation after warm-up.

## 1. Setup and imports

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').exists())
sys.path.insert(0, str(root))

from src.io import data_processing as dp
from src.io import data_visualization as dv
from src.io import run_workflow as rw
from src.io import validation_figures as vf
from src.io.runtime_paths import resolve_results_dir

pd.set_option('display.max_columns', 30, 'display.width', 160)

## 2. Run configuration

`WRITE_INPUT` stays `False` unless this run is about to be simulated. With `True`, section 4 overwrites the single input table read by the server.

In [ ]:
RUN_ID = 'T03'
STEP_SIZE = 0.01
WRITE_INPUT = False

config = rw.load_run_config(RUN_ID)
source = config.data_source
skip_time = source.skip_time
pd.Series({
    'Start': source.start_date,
    'End': source.end_date,
    'Time axis': source.time_axis,
    'Warm-up skipped [s]': skip_time,
    'Inputs': len(source.input_variables),
    'Outputs': len(source.output_variables),
}).to_frame('Value')

## 3. Measured data

Inputs and outputs of the window on the elapsed time axis.

In [ ]:
input_df, output_df = rw.load_measurements(config)
print(f'Inputs {input_df.shape}, outputs {output_df.shape}, end of window {input_df.index.max():.1f} s')
print('Sample interval [s]:')
display(pd.Series(input_df.index).diff().describe().round(3).to_frame('dt').T)
display(input_df.describe().T.round(3))
display(output_df.describe().T.round(3))

In [ ]:
fig = vf.plot_signals(input_df)

## 4. Simulation input

The table as it enters the simulation. It is only written with `WRITE_INPUT = True`.

In [ ]:
table_df = rw.simulation_table(input_df)
stop_time = float(table_df.index.max())
display(table_df.head())
print(f'Simulation setup: stop time {stop_time} s, interval 0.01 s, tolerance 1e-4, DASSL')

if WRITE_INPUT:
    dp.prepare_sim_input(config, table_df)
    print('Input table written.')
else:
    print('Input table not written.')

## 5. Stored simulation result

Simulate the model and store the result first. This section loads the latest stored attempt.

In [ ]:
result_path = rw.latest_result(RUN_ID)
display(rw.stored_run_summary(result_path).to_frame('Value'))

sim_df = rw.load_simulation(config, result_path, STEP_SIZE)
print(f'Simulated signals {sim_df.shape}, end {sim_df.index.max():.1f} s')

## 6. Comparison on the measured time base

System quantities are added and the simulation is sampled at the measured times. The shaded span is the skipped warm-up.

In [ ]:
sim_df, source_df = rw.align_comparison(config, sim_df, output_df)
fig = vf.plot_time_comparison(sim_df, source_df, skip_time)

In [ ]:
sim_eval, source_eval = sim_df.loc[skip_time:], source_df.loc[skip_time:]
print(f'Samples after warm-up: {len(source_eval)} of {len(source_df)}')

## 7. Error metrics

The signals are compared sample by sample after warm-up, without grouping.

In [ ]:
results = rw.compare_transient(sim_eval, source_eval)
results['metrics'].round(4)

## 8. Deviation over time

Simulation minus experiment.

In [ ]:
columns = [c for c in vf.TIME_COLUMNS if c in sim_eval and c in source_eval]
deviation = sim_eval[columns] - source_eval[columns]
fig = vf.plot_signals(deviation)